# Expected Shortfall (ES)

VaR has a major mathematical flaw: it is not a Coherent Risk Measure because it doesn't account for "Tail Risk": it only tells you where the door is, not how big the fire is behind it)

Expected Shortfall is simply the Conditional Mean.

The Heuristic: "If today is a 'Top 5% Bad Day,' what is the average loss I should expect?"

The Math: $ES_{\alpha} = E[X | X \le VaR_{\alpha}]$

In [16]:
import sys, os
import numpy as np
import pandas as pd
sys.path.append(os.path.abspath(os.path.join('..')))  # Add the parent directory to the path
from core.db_manager import RiskDBManager

# 1. Pull the data (Using our 'ID 2' logic from yesterday)
db = RiskDBManager()
query = """
    SELECT s.ticker, d.trade_date, d.adj_close
    FROM daily_metrics d
    JOIN securities s ON d.security_id = s.security_id
    ORDER BY d.trade_date ASC
"""
df = db.get_data(query)

# 2. Mechanistic Mapping: Pivot the table
# This puts Tickers as columns and Dates as rows
pivot_df = df.pivot(index='trade_date', columns='ticker', values='adj_close').astype(float)

# 3. Calculate Log Returns for the whole Portfolio
returns = np.log(pivot_df / pivot_df.shift(1)).dropna()

# 4. Calculate 95% Historical VaR (Per Ticker)
# axis=0 is the default; it calculates the 5th percentile for each column
var_95 = returns.quantile(0.05)

# 5. Calculate Expected Shortfall (Per Ticker)
# We use a mask to keep only the 'Bad' days for each specific asset
es_95 = returns[returns <= var_95].mean()

# 6. Formatting the Output (The Multi-Asset Print)
print("--- Risk Profile Audit ---")
for ticker in returns.columns:
    v = var_95[ticker]
    e = es_95[ticker]
    print(f"[{ticker}]")
    print(f"  95% VaR: {v:.4%}")
    print(f"  95% ES : {e:.4%}")
    print(f"  Tail Severity: {e - v:.4%}\n")


--- Risk Profile Audit ---
[AAPL]
  95% VaR: -2.7421%
  95% ES : -4.0702%
  Tail Severity: -1.3281%

[SPY]
  95% VaR: -1.5580%
  95% ES : -2.4303%
  Tail Severity: -0.8723%



c:\Users\beall\Documents\Risk Management\RQ-Core\core\db_manager.py:17: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(query, self.conn, params=params)


In [3]:
# Calculate 99% Historical VaR (The 1st Percentile / 5th worst day)
var_99 = np.percentile(returns, 1)

# Calculate 99% Expected Shortfall
es_99 = returns[returns <= var_99].mean()

print(f"99% Historical VaR: {var_99:.4%}")
print(f"99% Expected Shortfall: {es_99:.4%}")
print(f"99% Difference: {es_99 - var_99:.4%}")

99% Historical VaR: -4.9366%
99% Expected Shortfall: -6.2355%
99% Difference: -1.2989%


**LEPTOKURTOSIS (Fat Tails)** 

The ES/Var gap is widening from -1.37 to -1.56

95% Level: If you breach the wall, you lose an extra 1.37% on average.

99% Level: If you breach the wall, you lose an extra 1.56% on average.

**Artzner et al. (1999) properties of a "Coherent Risk Measure."**

VaR Fails: It fails the Subadditivity test. Sometimes, merging two portfolios can actually increase VaR, which is mathematically "sketchy" (diversification should reduce risk).

ES Succeeds: Expected Shortfall is Subadditive. It always rewards diversification.

By calculating that -6.50%, you are using a "Coherent" metric that tells you the true cost of a "worst-case" week for Apple.

In [7]:
tail_events.describe()

count    26.000000
mean     -0.040741
std       0.015518
min      -0.097013
25%      -0.042918
50%      -0.035149
75%      -0.032082
max      -0.027582
Name: returns, dtype: float64

In [8]:
returns.sort_values().head(26).mean()

np.float64(-0.04074095140048021)

In [9]:
var_95

np.float64(-0.02758196574946113)

In [11]:
returns.sort_values().iloc[25]

np.float64(-0.02758196574946113)

In [17]:
# Sort and look at the 'Worst' 26 days
tail_events = returns['AAPL'].sort_values().head(26)

print(f"Number of observations: {len(tail_events)}")
print(f"Best 'Worst' Day (The VaR): {tail_events.iloc[-1]:.4%}")
print(f"Worst 'Worst' Day: {tail_events.iloc[0]:.4%}")

Number of observations: 26
Best 'Worst' Day (The VaR): -2.7412%
Worst 'Worst' Day: -9.7013%
